In [1]:

%pip install pandas scikit-learn joblib


Note: you may need to restart the kernel to use updated packages.


In [2]:

import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")


Libraries imported successfully!


In [4]:

from pathlib import Path

print("Current working folder:")
print(Path.cwd())

print("\nCSV files in the current folder and its subfolders:")

for file in Path.cwd().rglob("*.csv"):
    print(file)


Current working folder:
c:\Users\Rakshitha Srinivasa\OneDrive\Desktop\Customer-Churn-Prediction\notebook

CSV files in the current folder and its subfolders:


In [5]:

data_path = Path("../data/Customer-Churn-Prediction.csv")

data = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", data.shape)

display(data.head())


Dataset loaded successfully!
Dataset shape: (7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [6]:

print("Dataset columns:")
print(data.columns.tolist())

print("\nFirst 5 rows:")
display(data.head())

print("\nChurn distribution:")
print(data["Churn Label"].value_counts())


Dataset columns:
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']

First 5 rows:


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices



Churn distribution:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64


In [7]:

data = data.dropna(subset=["Churn Label"]).copy()

y = data["Churn Label"].map({
    "No": 0,
    "Yes": 1
})

columns_to_drop = [
    "Churn Label",
    "Churn Value",
    "Churn Score",
    "Churn Reason",
    "CustomerID",
    "Count",
    "Country",
    "State",
    "City",
    "CLTV"
]

X = data.drop(columns=columns_to_drop, errors="ignore")

valid_rows = y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows].astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget values:")
print(y.value_counts())


Features shape: (7043, 23)
Target shape: (7043,)

Target values:
Churn Label
0    5174
1    1869
Name: count, dtype: int64


In [8]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


Training data: (5634, 23)
Testing data: (1409, 23)
Training target: (5634,)
Testing target: (1409,)


In [9]:

numeric_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numerical columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)


Numerical columns:
['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges']

Categorical columns:
['Lat Long', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Total Charges']


In [10]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Prepare numerical columns
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Prepare categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing methods
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

print("Data preprocessing prepared successfully!")


Data preprocessing prepared successfully!


In [11]:

from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train the model
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")


Logistic Regression model trained successfully!


In [12]:

# Predict churn for test customers
y_pred = model.predict(X_test)

print("First 10 actual values:")
print(y_test.head(10).tolist())

print("\nFirst 10 predicted values:")
print(y_pred[:10].tolist())


First 10 actual values:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

First 10 predicted values:
[0, 1, 0, 1, 0, 1, 1, 0, 0, 1]


In [13]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred, zero_division=0
)

recall = recall_score(
    y_test, y_pred, zero_division=0
)

f1 = f1_score(
    y_test, y_pred, zero_division=0
)

print("Logistic Regression Evaluation")
print("--------------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")


Logistic Regression Evaluation
--------------------------------
Accuracy:  0.7637
Precision: 0.5400
Recall:    0.7406
F1-score:  0.6246


In [14]:

from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Stayed", "Churned"],
    zero_division=0
))


              precision    recall  f1-score   support

      Stayed       0.89      0.77      0.83      1035
     Churned       0.54      0.74      0.62       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409



In [15]:

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

print("Confusion Matrix:")
print(cm)


Confusion Matrix:
[[799 236]
 [ 97 277]]


In [16]:

import joblib
from pathlib import Path

models_folder = Path("../models")
models_folder.mkdir(parents=True, exist_ok=True)

model_path = models_folder / "logistic_regression_model.joblib"

joblib.dump(model, model_path)

print("Model saved successfully!")
print("Location:", model_path.resolve())


Model saved successfully!
Location: C:\Users\Rakshitha Srinivasa\OneDrive\Desktop\Customer-Churn-Prediction\models\logistic_regression_model.joblib


In [17]:

import pandas as pd

results = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}])

results.to_csv(
    models_folder / "logistic_regression_metrics.csv",
    index=False
)

display(results)

print("Evaluation metrics saved successfully!")


,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.763662,0.539961,0.740642,0.624577


Evaluation metrics saved successfully!


In [18]:

from pathlib import Path

models_folder = Path("../models")

print("Checking saved files...\n")

for filename in [
    "logistic_regression_model.joblib",
    "logistic_regression_metrics.csv"
]:
    file_path = models_folder / filename

    if file_path.exists():
        print(f"FOUND: {file_path.resolve()}")
    else:
        print(f"MISSING: {file_path.resolve()}")


Checking saved files...

FOUND: C:\Users\Rakshitha Srinivasa\OneDrive\Desktop\Customer-Churn-Prediction\models\logistic_regression_model.joblib
FOUND: C:\Users\Rakshitha Srinivasa\OneDrive\Desktop\Customer-Churn-Prediction\models\logistic_regression_metrics.csv


In [19]:

import joblib

saved_model = joblib.load(
    "../models/logistic_regression_model.joblib"
)

test_predictions = saved_model.predict(X_test)

print("Saved model loaded successfully!")
print("First 10 predictions:", test_predictions[:10])


Saved model loaded successfully!
First 10 predictions: [0 1 0 1 0 1 1 0 0 1]


In [20]:

from pathlib import Path

print("Current notebook location:")
print(Path.cwd())

print("\nChecking saved files:")

for file in [
    Path("../models/logistic_regression_model.joblib"),
    Path("../models/logistic_regression_metrics.csv")
]:
    if file.exists():
        print("FOUND:", file.name)
    else:
        print("NOT FOUND:", file.name)


Current notebook location:
c:\Users\Rakshitha Srinivasa\OneDrive\Desktop\Customer-Churn-Prediction\notebook

Checking saved files:
FOUND: logistic_regression_model.joblib
FOUND: logistic_regression_metrics.csv


In [21]:

from pathlib import Path

print("Your current location:")
print(Path.cwd())

print("\nFiles in your project folder:")

for file in Path.cwd().rglob("logistic_regression*"):
    print(file)


Your current location:
c:\Users\Rakshitha Srinivasa\OneDrive\Desktop\Customer-Churn-Prediction\notebook

Files in your project folder:
c:\Users\Rakshitha Srinivasa\OneDrive\Desktop\Customer-Churn-Prediction\notebook\logistic_regression.ipynb
